# Q-Shield multimodal: visual + URL text + fusion

Compares three configurations on the same 21,998-sample joint validation set:

1. **Visual only** — the Phase-2 Siamese classifier already trained in notebook 06.
2. **Text only** — DistilBERT fine-tuned on the URLs decoded offline from each QR.
3. **Fusion** — late fusion of both branches (logit-level MLP).

Decoding is done **offline with pyzbar**: it produces a string from the QR matrix; it does not open or fetch the URL. Samples that fail to decode receive an `<UNDECODABLE>` token so the text branch can learn that signal too.

**Drive layout assumed**: the Drive folder `Proyecto_Quishing_Detection_Nicolas` holds **data and checkpoints only** (`*.zip`, `classifier_v3_phase2.pth`, etc.). The Python source (`qshield` package) lives in the GitHub repo and is pulled fresh each session into `/content/qshield_repo`.

## 1. Setup

Two locations:
- `BASE` — Drive folder with data zips and checkpoints.
- `REPO` — local clone of the GitHub repo (source code lives here).

If you have not pushed the latest `qshield` package to GitHub yet, the import below will fail. Either (a) push first, or (b) drop a copy of `src/qshield` into Drive at `BASE/qshield_src/qshield/` and the cell will fall back to that.

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

BASE = '/content/drive/MyDrive/Proyecto_Quishing_Detection_Nicolas'
REPO = '/content/qshield_repo'
REPO_GIT = 'https://github.com/nicolasllerenas/Multimodal-Quishing-Detection-Framework.git'
WORK = '/content/qshield_work' if IN_COLAB else os.path.join(BASE, 'data', 'work')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    assert os.path.exists(BASE), f'Drive folder not found: {BASE}'

    # Source code: pull a fresh clone (or update if already there)
    if not os.path.exists(REPO):
        !git clone -q $REPO_GIT $REPO
    else:
        !cd $REPO && git pull -q

    # Runtime deps. Don't pin transformers — Colab ships a recent torch and the
    # default transformers wheel is the safest. pyzbar needs libzbar0 from apt.
    !apt-get -qq install libzbar0
    !pip install -q transformers pyzbar
else:
    REPO = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('.')), '..'))
    if not os.path.basename(REPO).endswith('Detection-Framework'):
        REPO = os.path.dirname(os.getcwd())

# Source location: prefer the GitHub clone, fall back to a Drive copy if the user
# has not pushed the package yet.
candidates = [
    os.path.join(REPO, 'src'),                  # github clone, normal case
    os.path.join(BASE, 'qshield_src'),          # fallback: rsync src/qshield -> Drive
]
src_dir = next((p for p in candidates if os.path.isdir(os.path.join(p, 'qshield'))), None)
if src_dir is None:
    raise RuntimeError(
        'qshield package not found.\n'
        f'Looked in: {candidates}\n'
        'Either (a) git push the local changes so the GitHub clone has src/qshield, '
        f'or (b) copy your local src/qshield into {os.path.join(BASE, "qshield_src", "qshield")}.'
    )
sys.path.insert(0, src_dir)
os.makedirs(WORK, exist_ok=True)

print('BASE :', BASE)
print('REPO :', REPO)
print('SRC  :', src_dir)
print('WORK :', WORK)

In [ ]:
import qshield
from qshield.utils.seeds import set_all_seeds
import torch

DATA_SEED = 42
set_all_seeds(DATA_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Set this to a value in (0, 1] to subsample for a quick smoke test before
# committing to the 90-min full run. SAMPLE_FRACTION = 0.04 covers the whole
# pipeline in roughly 10 min on T4 (~4k samples).
SAMPLE_FRACTION = 1.0

print('qshield', qshield.__version__, '| device:', device,
      '| SAMPLE_FRACTION:', SAMPLE_FRACTION)


## 2. Load data and build splits

Same canonical 80/20 split used by every other notebook (DATA_SEED=42), so the val set is byte-identical to the visual-only evaluation.

In [ ]:
from qshield.data import trad, cic, splits
from qshield.data.classify import ClassifyDataset

trad_qr, trad_labels = trad.load(BASE, WORK)
cic_b, cic_m = cic.load(BASE, WORK, n_per_class=50000, data_seed=DATA_SEED)

(qr_tr, lab_tr, _), (qr_val, lab_val, _) = splits.trad_split(trad_qr, trad_labels, DATA_SEED)
(cic_b_tr, cic_m_tr), (cic_b_val, cic_m_val) = splits.cic_split(cic_b, cic_m)

# Optional smoke-test subsample. Applied AFTER the canonical split so the
# subsample stays stratified and reproducible.
if SAMPLE_FRACTION < 1.0:
    import random
    rng = random.Random(DATA_SEED)
    def take(seq, frac):
        if isinstance(seq, list):
            n = max(1, int(len(seq) * frac))
            return rng.sample(seq, n)
        n = max(1, int(len(seq) * frac))
        idx = rng.sample(range(len(seq)), n)
        return seq[idx], idx
    qr_tr, idx_tr = take(qr_tr, SAMPLE_FRACTION); lab_tr = lab_tr[idx_tr]
    qr_val, idx_val = take(qr_val, SAMPLE_FRACTION); lab_val = lab_val[idx_val]
    cic_b_tr = take(cic_b_tr, SAMPLE_FRACTION)
    cic_m_tr = take(cic_m_tr, SAMPLE_FRACTION)
    cic_b_val = take(cic_b_val, SAMPLE_FRACTION)
    cic_m_val = take(cic_m_val, SAMPLE_FRACTION)
    print(f'Subsampled to {SAMPLE_FRACTION:.0%} of each split.')

print(f'Train: trad={len(qr_tr):,}  cic_b={len(cic_b_tr):,}  cic_m={len(cic_m_tr):,}')
print(f'Val:   trad={len(qr_val):,}  cic_b={len(cic_b_val):,}  cic_m={len(cic_m_val):,}')

train_ds = ClassifyDataset(qr_tr, lab_tr, cic_b_tr, cic_m_tr, augment=False, return_index=True)
val_ds = ClassifyDataset(qr_val, lab_val, cic_b_val, cic_m_val, augment=False, return_index=True)
print(f'Total train: {len(train_ds):,}  val: {len(val_ds):,}')


## 3. Decode QRs (offline)

Walks every train and val sample once, decodes it with pyzbar, and stores the result in a JSON cache on Drive. Re-running this cell skips already-cached samples, so it is idempotent. The full corpus takes roughly 15 minutes on T4.

In [ ]:
from qshield.data.decode import build_cache_for_dataset, UNDECODABLE

cache_path = os.path.join(BASE, 'url_cache.json')
url_cache = build_cache_for_dataset(train_ds, cache_path=cache_path)
url_cache = build_cache_for_dataset(val_ds,   cache_path=cache_path)
print(f'Cached URLs: {len(url_cache):,}')

n_unk = sum(1 for v in url_cache.values() if v == UNDECODABLE)
print(f'Decode failures: {n_unk:,}  ({n_unk / len(url_cache):.1%})')

### 3.1 Decode-failure breakdown by source

If Trad's failure rate is much higher than CIC's, the text branch will end up
specialising on CIC and the fusion will leak dataset identity. Report it
explicitly so the paper can describe the regime instead of hiding it.

In [ ]:
from collections import Counter
from qshield.data.decode import UNDECODABLE

per_src = Counter()
for k, v in url_cache.items():
    src = k.split(':', 1)[0]
    per_src[(src, v == UNDECODABLE)] += 1

print(f'{"source":<10} {"decoded":>10} {"undecodable":>12} {"failure rate":>14}')
print('-' * 50)
for src in ('trad', 'cic_b', 'cic_m'):
    ok = per_src.get((src, False), 0)
    bad = per_src.get((src, True), 0)
    total = ok + bad
    rate = bad / total if total else 0.0
    print(f'{src:<10} {ok:>10,} {bad:>12,} {rate:>13.1%}')


## 4. Visual branch — load the existing Phase-2 classifier

We do not retrain the visual branch here; we reuse `classifier_v3_phase2.pth` from notebook 06. To compare against the seed-7 ensemble member, point `VISUAL_CKPT` at `classifier_v3_seed7_phase2.pth`.

In [ ]:
from qshield.models.visual import load_classifier

VISUAL_CKPT = os.path.join(BASE, 'classifier_v3_phase2.pth')
visual = load_classifier(VISUAL_CKPT, device)
print('Visual params:', sum(p.numel() for p in visual.parameters()))

## 5. Text branch — fine-tune DistilBERT on the decoded URLs

DistilBERT is the safe default; for tighter latency budgets swap `TEXT_MODEL = 'bert-tiny'` (4M params).

In [ ]:
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from torch.utils.data import DataLoader
from functools import partial

from qshield.data.url_dataset import MultimodalDataset, URLOnlyDataset
from qshield.models.text import URLClassifier, URLTokenizer, collate_multimodal
from qshield.models.losses import FocalLoss
from qshield.training.text_train import train as train_text


TEXT_MODEL = 'distilbert'
tokenizer = URLTokenizer(model_name=TEXT_MODEL, max_length=96)
text_model = URLClassifier(model_name=TEXT_MODEL).to(device)

text_tr = URLOnlyDataset(train_ds, url_cache)
text_val = URLOnlyDataset(val_ds, url_cache)
mm_train = MultimodalDataset(train_ds, url_cache)
mm_val = MultimodalDataset(val_ds, url_cache)

collate = partial(collate_multimodal, tokenizer=tokenizer)
train_loader_t = DataLoader(text_tr, batch_size=128, shuffle=True,
                            num_workers=0, collate_fn=collate)
val_loader_t = DataLoader(text_val, batch_size=256, shuffle=False,
                          num_workers=0, collate_fn=collate)

text_model, text_history = train_text(
    text_model, train_loader_t, val_loader_t,
    criterion=FocalLoss(alpha=0.5, gamma=2.0),
    epochs=3, lr=2e-5, device=device,
)
TEXT_CKPT = os.path.join(BASE, 'text_v1_distilbert.pth')
torch.save(text_model.state_dict(), TEXT_CKPT)
print('Saved:', TEXT_CKPT)


## 6. Late fusion

Run both branches once over train + val to cache their logits. Then train a tiny MLP on `[visual_logit, text_logit, decode_failed_flag]`.

In [ ]:
from qshield.models.fusion import LogitFusion
from qshield.training.fusion_train import precompute_logits, train as train_fusion

train_feats = precompute_logits(visual, text_model, tokenizer, mm_train,
                                batch_size=128, device=device)
val_feats = precompute_logits(visual, text_model, tokenizer, mm_val,
                              batch_size=128, device=device)

fusion = LogitFusion(hidden=16, use_decode_flag=True)
fusion, fusion_history = train_fusion(
    fusion, train_feats, val_feats,
    epochs=20, lr=1e-3, device=device,
)
FUSION_CKPT = os.path.join(BASE, 'fusion_v1_logit_mlp.pth')
torch.save(fusion.state_dict(), FUSION_CKPT)
print('Saved:', FUSION_CKPT)

## 7. Final comparison

Reports visual-only / text-only / fusion on the same val set, plus calibration metrics (ECE, Brier). Numbers go into `eval_multimodal.json` on Drive and feed Table III of the paper.

In [ ]:
import json
import numpy as np
from qshield.eval.metrics import report, pretty

v_logits, t_logits, flags, labels = val_feats
labels_np = labels.numpy().astype(int)

v_probs = torch.sigmoid(v_logits).numpy()
t_probs = torch.sigmoid(t_logits).numpy()

with torch.no_grad():
    fused_logits = fusion(v_logits, t_logits, flags).squeeze(1)
fused_probs = torch.sigmoid(fused_logits).numpy()

results = {
    'visual_only': report(v_probs, labels_np),
    'text_only':   report(t_probs, labels_np),
    'fusion':      report(fused_probs, labels_np),
}
for name, r in results.items():
    print(pretty(r, name))

out_path = os.path.join(BASE, 'eval_multimodal.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved:', out_path)